# Pregunta 3 — CFA, SEM y comparación por género

**Métodos Cuantitativos de Investigación en Negocios — Tarea 2 (2026)**

In [1]:
# Instalación condicional de dependencias para Google Colab.
# En ejecución local no instala nada si los paquetes ya están disponibles.
import importlib, subprocess, sys

def instalar_si_falta(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Instalando {package_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

instalar_si_falta('pyreadstat', 'pyreadstat')
instalar_si_falta('semopy', 'semopy')

## Enunciado de la pregunta 3

**3.** Como recién llegado a la consultora antes mencionada, le han incluido en un segundo proyecto, relacionado con los efectos de las condiciones de trabajo sobre la satisfacción y permanencia de los trabajadores. En este contexto, se han considerado los siguientes constructos relevantes para la investigación:

- **Actitud hacia compañeros de trabajo:** Actitud del empleado hacia/con sus compañeros de trabajo con los que interactúa regularmente.
- **Percepción del ambiente de trabajo:** Percepción del trabajador respecto a su lugar de trabajo.
- **Satisfacción con el trabajo:** Medición del nivel de satisfacción del empleado con su trabajo.
- **Compromiso organizacional:** Grado en que el empleado se identifica y siente parte de la empresa.
- **Intención de permanecer en el trabajo:** Grado en el que el trabajador pretende continuar trabajando para la empresa.
- **Características personales:** Edad, Género, y experiencia en años del trabajador.

Cada uno de estos constructos fue medido en escalas de múltiples ítems que se describen a continuación, y que fueron aplicadas en un cuestionario a 400 empleados de diversas empresas del sector de telecomunicaciones, disponibles en el archivo `Laboral.sav`.

![Tabla de constructos e ítems de la pregunta 3](https://github.com/sebabecerra/Metodos-Cuantitativos-DEN/blob/main/Tarea2/assets/p3_tabla_constructos.png?raw=1)

Las escalas específicas de cada ítem se encuentran detalladas en el archivo de datos `Laboral.sav`.

Gracias a sus conocimientos en el área de comportamiento organizacional, usted ha identificado dos posibles modelos que expliquen la relación entre las condiciones de trabajo percibidas por el trabajador, y su satisfacción e intenciones de permanecer en la empresa. Estos modelos difieren principalmente en si el efecto de la precepción de ambiente de trabajo y la actitud hacia los compañeros de trabajo tienen efectos completamente mediados o parcialmente mediados sobre la intención de permanecer en la empresa.

Cada uno de los modelos se describe a continuación:

![Modelo completamente mediado](https://github.com/sebabecerra/Metodos-Cuantitativos-DEN/blob/main/Tarea2/assets/p3_modelo_completamente_mediado.png?raw=1)

![Modelo parcialmente mediado](https://github.com/sebabecerra/Metodos-Cuantitativos-DEN/blob/main/Tarea2/assets/p3_modelo_parcialmente_mediado.png?raw=1)

Su labor consiste en determinar:

**a)** Qué tan bueno es el modelo de medición, incluyendo medidas de confiabilidad y validez de escalas.  
**b)** Determinar cuál de los dos modelos representa de mejor forma los datos recopilados.  
**c)** Para el modelo identificado como superior, evalúe si las relaciones entre las distintas variables se comportan de la misma forma entre hombres y mujeres.  
**d)** Explique sus resultados y conclusiones.

## Respuesta

La pregunta se resuelve en dos etapas: primero se evalúa el modelo de medición mediante CFA, confiabilidad y validez; luego se comparan los modelos estructurales completamente mediado y parcialmente mediado mediante SEM. Finalmente, el modelo superior se analiza por género para evaluar si las relaciones se comportan de forma similar entre hombres y mujeres.


In [2]:
from pathlib import Path
from urllib.request import urlretrieve, urlopen
import ssl
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

# Base requerida para este notebook.
# Funciona localmente y también en Google Colab abierto desde GitHub.
REQUIRED_DATA_FILE = 'Laboral.sav'
RAW_DATA_BASE = 'https://raw.githubusercontent.com/sebabecerra/Metodos-Cuantitativos-DEN/main/Tarea2/data'

def encontrar_o_descargar_base(filename):
    candidatas = [
        Path.cwd() / 'data',                 # ejecución desde Tarea2
        Path.cwd().parent / 'data',          # ejecución desde Tarea2/notebooks
        Path.cwd() / 'Tarea2' / 'data',      # ejecución desde raíz del repo o Colab
        Path.cwd()                           # respaldo: base junto al notebook
    ]
    for ruta in candidatas:
        if (ruta / filename).exists():
            return ruta

    # Si Colab no tiene la carpeta del repo clonada, se descarga la base desde GitHub.
    destino_dir = Path.cwd() / 'Tarea2' / 'data'
    destino_dir.mkdir(parents=True, exist_ok=True)
    destino = destino_dir / filename
    url = f'{RAW_DATA_BASE}/{filename}'
    print(f'Base {filename} no encontrada localmente. Descargando desde GitHub...')
    try:
        urlretrieve(url, destino)
    except Exception as error:
        # Respaldo útil en algunos entornos locales con problemas de certificados SSL.
        # En Colab normalmente no se activa, pero evita fallas si Python no encuentra certificados.
        print('Descarga estándar falló; intentando respaldo SSL...')
        contexto = ssl._create_unverified_context()
        with urlopen(url, context=contexto) as respuesta, open(destino, 'wb') as archivo:
            archivo.write(respuesta.read())
    return destino_dir

DATA_DIR = encontrar_o_descargar_base(REQUIRED_DATA_FILE)
print('Directorio de datos:', DATA_DIR)
print('Base usada:', DATA_DIR / REQUIRED_DATA_FILE)


Directorio de datos: /Users/sbc/projects/Metodos-Cuantitativos-DEN/Tarea2/data
Base usada: /Users/sbc/projects/Metodos-Cuantitativos-DEN/Tarea2/data/Laboral.sav


## 3.a) Modelo de medición

Antes de comparar los modelos estructurales, se debe verificar que los constructos estén bien medidos. Esta etapa corresponde a un **Análisis Factorial Confirmatorio (CFA)**. La idea es confirmar que cada grupo de ítems mide el constructo teórico que corresponde.

En este problema hay cinco constructos latentes:

- $AC$: actitud hacia compañeros de trabajo.
- $PA$: percepción del ambiente de trabajo.
- $ST$: satisfacción con el trabajo.
- $CO$: compromiso organizacional.
- $IP$: intención de permanecer en el trabajo.

Cada constructo se mide con cuatro indicadores observados. La forma general de una ecuación de medición es:

$$x_{ij}=\lambda_{ij}\eta_j+\varepsilon_{ij}$$

donde $x_{ij}$ es el ítem observado, $\eta_j$ es el constructo latente, $\lambda_{ij}$ es la carga factorial y $\varepsilon_{ij}$ es el error de medición del ítem.

Aplicado a esta tarea, el modelo de medición se escribe así:

$$AC =~ AC1 + AC2 + AC3 + AC4$$
$$PA =~ PA1 + PA2 + PA3 + PA4$$
$$ST =~ ST1 + ST2 + ST3 + ST4$$
$$CO =~ CO1 + CO2 + CO3 + CO4$$
$$IP =~ IP1 + IP2 + IP3 + IP4$$

En esta etapa se revisa:

1. **Ajuste global del modelo de medición:** si la estructura de cinco factores reproduce razonablemente la matriz de covarianzas observada.
2. **Cargas factoriales:** si los ítems se asocian fuertemente con su constructo.
3. **Confiabilidad:** si los ítems de cada constructo son consistentes entre sí.
4. **Validez convergente:** si los indicadores de un mismo constructo comparten suficiente varianza.
5. **Validez discriminante:** si los constructos son distintos entre sí y no están midiendo exactamente lo mismo.

Este paso es necesario porque no tendría sentido comparar relaciones causales entre constructos si antes no se verifica que esos constructos están medidos de forma razonable.


In [3]:
from semopy import Model, calc_stats
df,meta=pyreadstat.read_sav(DATA_DIR/'Laboral.sav',apply_value_formats=False)
constructos={'AC':['AC1','AC2','AC3','AC4'],'PA':['PA1','PA2','PA3','PA4'],'ST':['ST1','ST2','ST3','ST4'],'CO':['CO1','CO2','CO3','CO4'],'IP':['IP1','IP2','IP3','IP4']}
obs=sum(constructos.values(),[]); datos=df[obs+['GENERO']].dropna()
print('Casos:',len(datos),'Hombres:',sum(datos.GENERO==0),'Mujeres:',sum(datos.GENERO==1))

Casos: 400 Hombres: 200 Mujeres: 200


In [4]:
# Estimación del CFA puro: cinco factores libremente correlacionados.
# Esta celda responde cuantitativamente si cada ítem carga en el constructo que corresponde.
medicion = '\n'.join(f"{k} =~ {' + '.join(v)}" for k, v in constructos.items())
latentes = list(constructos)
covarianzas = []
for i, a in enumerate(latentes):
    for b in latentes[i+1:]:
        covarianzas.append(f'{a} ~~ {b}')

cfa_desc = medicion + '\n' + '\n'.join(covarianzas)
modelo_cfa = Model(cfa_desc)
modelo_cfa.fit(datos)
ajuste_cfa = calc_stats(modelo_cfa).iloc[0]
param_cfa = modelo_cfa.inspect(std_est=True)

cargas_cfa = param_cfa[(param_cfa.op == '~') & (param_cfa.rval.isin(constructos.keys()))][
    ['rval', 'lval', 'Estimate', 'Est. Std', 'Std. Err', 'z-value', 'p-value']
].copy()
cargas_cfa.columns = ['Constructo', 'Ítem', 'Carga no est.', 'Carga est.', 'EE', 'z', 'p']
cargas_cfa['orden'] = cargas_cfa['Constructo'].map({k: i for i, k in enumerate(constructos)}) * 10 + cargas_cfa['Ítem'].str.extract(r'(\d+)').astype(int)[0]
cargas_cfa = cargas_cfa.sort_values('orden').drop(columns='orden')

print('Ajuste global del CFA de medición')
print(pd.Series({
    'chi2': ajuste_cfa.chi2,
    'gl': ajuste_cfa.DoF,
    'CFI': ajuste_cfa.CFI,
    'TLI': ajuste_cfa.TLI,
    'RMSEA': ajuste_cfa.RMSEA
}).round(4).to_string())

print('\nCargas factoriales estandarizadas por ítem')
print(cargas_cfa.round(4).to_string(index=False))


Ajuste global del CFA de medición
chi2     220.3091
gl       160.0000
CFI        0.9850
TLI        0.9822
RMSEA      0.0307

Cargas factoriales estandarizadas por ítem
Constructo Ítem  Carga no est.  Carga est.        EE          z    p
        AC  AC1         1.0000      0.8219         -          -    -
        AC  AC2         1.2363      0.8203   0.06723  18.388501  0.0
        AC  AC3         1.0375      0.8372  0.054989  18.866634  0.0
        AC  AC4         1.1467      0.8155  0.062832  18.250812  0.0
        PA  PA1         1.0000      0.6917         -          -    -
        PA  PA2         1.0336      0.8040  0.073363  14.089158  0.0
        PA  PA3         0.8204      0.7783  0.059766  13.726961  0.0
        PA  PA4         0.9137      0.8229  0.063741  14.335176  0.0
        ST  ST1         1.0000      0.7371         -          -    -
        ST  ST2         1.0226      0.7368  0.081086  12.611086  0.0
        ST  ST3         0.9294      0.6968  0.076859  12.092472  0.0
    

### Confiabilidad, validez convergente y validez discriminante

Después de estimar el modelo de medición, se evalúa la calidad de cada escala.

### Confiabilidad

La confiabilidad indica si los ítems de un constructo son consistentes entre sí. Se reportan dos medidas:

- **Alfa de Cronbach:** mide consistencia interna bajo el supuesto de que los ítems son relativamente equivalentes.
- **Confiabilidad compuesta (CR):** usa las cargas factoriales del CFA, por lo que es más coherente con modelos de ecuaciones estructurales.

Como regla práctica, valores de alfa y CR iguales o superiores a 0.70 se consideran adecuados.

### Validez convergente

La validez convergente evalúa si los ítems que deberían medir un mismo constructo efectivamente comparten suficiente varianza. Se usa la **varianza media extraída (AVE)**:

$$AVE = \frac{\sum \lambda_i^2}{k}$$

Un AVE mayor o igual a 0.50 indica que el constructo explica al menos la mitad de la varianza de sus indicadores.

### Validez discriminante

La validez discriminante evalúa si los constructos son distintos entre sí. Se revisa con:

- **Correlaciones latentes:** no deberían ser excesivamente altas.
- **Fornell–Larcker:** la raíz cuadrada del AVE de cada constructo debe superar sus correlaciones con los demás constructos; así cada constructo comparte más varianza con sus propios ítems que con los otros conceptos.
- **HTMT:** valores menores a 0.85 suelen considerarse evidencia de discriminación adecuada.

Si una escala tiene buena confiabilidad, AVE suficiente y discriminación adecuada, se puede usar con más confianza en el modelo estructural.


A modo de ilustración del cálculo de CR y AVE, para $AC$ las cargas estandarizadas del CFA son $\lambda=(0.8219,\,0.8203,\,0.8372,\,0.8155)$, con $\sum\lambda_i=3.2949$ y $\sum(1-\lambda_i^2)=1.2857$. Reemplazando:

$$CR=\frac{(3.2949)^2}{(3.2949)^2+1.2857}=\frac{10.856}{12.142}=0.894,\qquad AVE=\frac{\sum\lambda_i^2}{4}=0.679.$$

In [5]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
filas=[]
for c,its in constructos.items():
 r=cargas_cfa[cargas_cfa['Constructo'].eq(c)].set_index('Ítem').reindex(its)['Carga est.'].astype(float)
 cr=r.sum()**2/(r.sum()**2+(1-r**2).sum()); ave=(r**2).mean()
 filas.append([c,alpha(datos[its]),cr,ave,np.sqrt(ave),r.min(),r.max()])
calidad=pd.DataFrame(filas,columns=['Constructo','Alfa','CR','AVE','raíz AVE','Carga mínima','Carga máxima'])
print(calidad.round(4).to_string(index=False))

# Correlaciones latentes implicadas por el propio CFA (estandarizadas).
# Se usan estas, y no puntajes factoriales estimados, porque son las que el modelo reproduce.
lat=param_cfa[(param_cfa.op=='~~')&param_cfa.lval.isin(constructos)&param_cfa.rval.isin(constructos)&(param_cfa.lval!=param_cfa.rval)]
print('\nCorrelaciones latentes implicadas por el CFA:')
for _,r in lat.iterrows(): print(f"{r.lval}-{r.rval}: {r['Est. Std']:.3f}")
max_corr=lat['Est. Std'].abs().max(); min_rave=calidad['raíz AVE'].min()
print(f'\nFornell–Larcker: mínima raíz de AVE = {min_rave:.3f} > correlación latente máxima = {max_corr:.3f} -> se cumple')

def htmt(a,b):
 A,B=constructos[a],constructos[b]; R=datos[A+B].corr().abs(); h=R.loc[A,B].values.mean(); ma=R.loc[A,A].values[np.triu_indices(4,1)].mean(); mb=R.loc[B,B].values[np.triu_indices(4,1)].mean(); return h/np.sqrt(ma*mb)
print('\nHTMT:')
for i,a in enumerate(constructos):
 for b in list(constructos)[i+1:]: print(f'{a}-{b}: {htmt(a,b):.3f}')

Constructo   Alfa     CR    AVE  raíz AVE  Carga mínima  Carga máxima
        AC 0.8908 0.8941 0.6786    0.8238        0.8155        0.8372
        PA 0.8474 0.8576 0.6019    0.7758        0.6917        0.8229
        ST 0.8112 0.8115 0.5185    0.7201        0.6968        0.7371
        CO 0.8227 0.8342 0.5638    0.7509        0.5836        0.8854
        IP 0.8863 0.8900 0.6698    0.8184        0.7410        0.8641

Correlaciones latentes implicadas por el CFA:
AC-PA: 0.257
AC-ST: 0.025
AC-CO: 0.307
AC-IP: 0.308
PA-ST: 0.228
PA-CO: 0.497
PA-IP: 0.562
ST-CO: 0.190
ST-IP: 0.214
CO-IP: 0.553

Fornell–Larcker: mínima raíz de AVE = 0.720 > correlación latente máxima = 0.562 -> se cumple

HTMT:
AC-PA: 0.260
AC-ST: 0.044
AC-CO: 0.275
AC-IP: 0.310
PA-ST: 0.231
PA-CO: 0.495
PA-IP: 0.569
ST-CO: 0.193
ST-IP: 0.217
CO-IP: 0.501


### Respuesta cuantitativa del modelo de medición

El modelo de medición presenta buen ajuste global. El CFA de cinco factores correlacionados obtuvo:

$$\chi^2(160)=220.3091,\quad CFI=0.9850,\quad TLI=0.9822,\quad RMSEA=0.0307$$

Estos valores son favorables: CFI y TLI están sobre 0.95 y RMSEA está bajo 0.06. Por lo tanto, la estructura de medición propuesta reproduce adecuadamente los datos observados.

La tabla de cargas factoriales muestra cuánto se asocia cada ítem con su constructo latente. Las cargas estandarizadas van desde 0.5836 hasta 0.8854. La mayoría de los ítems supera 0.70. Los ítems más bajos son CO1 = 0.5836, CO3 = 0.6571, PA1 = 0.6917 y ST3 = 0.6968, pero ninguno cae a un nivel problemático extremo y todos pertenecen a escalas con confiabilidad compuesta adecuada.

La calidad agregada por constructo es la siguiente:

| Constructo | Alfa | CR | AVE | Carga mínima | Carga máxima |
|---|---:|---:|---:|---:|---:|
| AC | 0.8908 | 0.8941 | 0.6786 | 0.8155 | 0.8372 |
| PA | 0.8474 | 0.8576 | 0.6019 | 0.6917 | 0.8229 |
| ST | 0.8112 | 0.8115 | 0.5185 | 0.6968 | 0.7371 |
| CO | 0.8227 | 0.8341 | 0.5638 | 0.5836 | 0.8854 |
| IP | 0.8863 | 0.8900 | 0.6698 | 0.7410 | 0.8641 |

La confiabilidad es adecuada porque todos los alfa de Cronbach son superiores a 0.80 y todas las confiabilidades compuestas son superiores a 0.81. La validez convergente también es adecuada porque todos los AVE son mayores a 0.50.

La validez discriminante es satisfactoria. El HTMT máximo es 0.569 para PA--IP, muy por debajo del criterio de 0.85. Además, la mayor correlación latente observada es 0.562 entre PA e IP, lo que indica asociación, pero no redundancia entre constructos.

En conclusión, el modelo de medición es bueno: presenta ajuste global adecuado, cargas factoriales mayoritariamente altas, confiabilidad suficiente, validez convergente y validez discriminante. Por eso es metodológicamente razonable avanzar a la comparación de los modelos estructurales completo y parcial.

### Conclusión 3.a

El modelo de medición es adecuado. Cuantitativamente, el ajuste global del CFA es bueno porque $CFI=0.9850$, $TLI=0.9822$ y $RMSEA=0.0307$. Además, todas las escalas presentan confiabilidad suficiente, con alfa entre 0.8112 y 0.8908, y confiabilidad compuesta entre 0.8115 y 0.8941.

La validez convergente también se cumple porque todos los AVE son mayores a 0.50. La validez discriminante es aceptable porque el HTMT máximo es 0.569, inferior al criterio de 0.85. Aunque algunos ítems tienen cargas algo más bajas, como CO1 = 0.5836, todas las escalas mantienen indicadores globales adecuados.

Por lo tanto, se concluye que el modelo de medición es suficientemente bueno para continuar con la comparación de los modelos estructurales completamente mediado y parcialmente mediado.



## 3.b) Comparación de los modelos estructurales

Una vez validado el modelo de medición, se comparan los dos modelos estructurales que aparecen en las figuras del enunciado.

### Modelo completamente mediado

El modelo completamente mediado supone que la percepción del ambiente de trabajo ($PA$) y la actitud hacia compañeros ($AC$) no afectan directamente la intención de permanecer ($IP$). Sin embargo, según la figura original, $PA$ y $AC$ sí predicen tanto satisfacción ($ST$) como compromiso organizacional ($CO$), y además $ST$ predice tanto $CO$ como $IP$.

Sus ecuaciones estructurales son:

$$ST = \gamma_1 PA + \gamma_2 AC + \zeta_{ST}$$
$$CO = \gamma_3 PA + \gamma_4 AC + eta_1 ST + \zeta_{CO}$$
$$IP = eta_2 ST + eta_3 CO + \zeta_{IP}$$

La lógica es: las condiciones de trabajo inciden en satisfacción y compromiso; satisfacción incide en compromiso e intención de permanencia; y compromiso incide en permanencia. No hay caminos directos desde $PA$ o $AC$ hacia $IP$.

### Modelo parcialmente mediado

El modelo parcialmente mediado mantiene los caminos anteriores, pero agrega efectos directos desde $PA$ y $AC$ hacia $IP$. Esto permite que el ambiente laboral y la relación con compañeros influyan en la intención de permanecer no solo a través de satisfacción y compromiso, sino también directamente.

Sus ecuaciones estructurales son:

$$ST = \gamma_1 PA + \gamma_2 AC + \zeta_{ST}$$
$$CO = \gamma_3 PA + \gamma_4 AC + eta_1 ST + \zeta_{CO}$$
$$IP = \gamma_5 PA + \gamma_6 AC + eta_2 ST + eta_3 CO + \zeta_{IP}$$

La diferencia clave está en la tercera ecuación: el modelo parcial agrega $PA
ightarrow IP$ y $AC
ightarrow IP$.

### Cómo se decide cuál modelo es mejor

Los modelos son **anidados**, porque el modelo completamente mediado es una versión restringida del parcialmente mediado. Por eso se puede comparar la pérdida de ajuste al imponer las restricciones $PA
ightarrow IP=0$ y $AC
ightarrow IP=0$.

Se revisan dos tipos de evidencia: índices de ajuste global (CFI, TLI, RMSEA, AIC y BIC) y diferencia de chi-cuadrado. Si $\Delta\chi^2$ es significativa, el modelo parcial mejora el ajuste de forma estadísticamente relevante.

In [6]:
med='\n'.join(f"{k} =~ {' + '.join(v)}" for k,v in constructos.items())
# Especificación corregida según las figuras del enunciado:
# Completo: PA y AC predicen ST y CO; ST predice CO e IP; CO predice IP.
# Parcial: agrega caminos directos PA->IP y AC->IP.
completo=med+'\nST ~ PA + AC\nCO ~ PA + AC + ST\nIP ~ ST + CO'
parcial=med+'\nST ~ PA + AC\nCO ~ PA + AC + ST\nIP ~ PA + AC + ST + CO'
def ajustar(desc,d):
    m=Model(desc); m.fit(d); return m,calc_stats(m).iloc[0],m.inspect(std_est=True)
mc,fc,pc=ajustar(completo,datos); mp,fp,pp=ajustar(parcial,datos)

# AIC y BIC calculados sobre el estadístico chi2: AIC = chi2 + 2k, BIC = chi2 + k·ln(n).
n=len(datos)
def criterios(m,f):
    k=len(m.param_vals); return k, f.chi2+2*k, f.chi2+k*np.log(n)
kc,aic_c,bic_c=criterios(mc,fc); kp,aic_p,bic_p=criterios(mp,fp)

tabla=pd.DataFrame({'Completo':[fc.chi2,fc.DoF,fc.CFI,fc.TLI,fc.RMSEA,kc,aic_c,bic_c],
                    'Parcial':[fp.chi2,fp.DoF,fp.CFI,fp.TLI,fp.RMSEA,kp,aic_p,bic_p]},
                   index=['chi2','gl','CFI','TLI','RMSEA','k (parámetros)','AIC (chi2+2k)','BIC (chi2+k·ln n)'])
print(tabla.round(4).to_string())

dchi=fc.chi2-fp.chi2; ddf=fc.DoF-fp.DoF
print(f'\nDiferencia chi-cuadrado = {fc.chi2:.4f} - {fp.chi2:.4f} = {dchi:.4f}, gl = {ddf:.0f}, p = {stats.chi2.sf(dchi,ddf):.4g}')
print(f'chi2 crítico (95%, gl={ddf:.0f}) = {stats.chi2.ppf(.95,ddf):.4f}')

                   Completo   Parcial
chi2               267.1340  220.3087
gl                 162.0000  160.0000
CFI                  0.9738    0.9850
TLI                  0.9693    0.9822
RMSEA                0.0403    0.0307
k (parámetros)      48.0000   50.0000
AIC (chi2+2k)      363.1340  320.3087
BIC (chi2+k·ln n)  554.7243  519.8819

Diferencia chi-cuadrado = 267.1340 - 220.3087 = 46.8253, gl = 2, p = 6.792e-11
chi2 crítico (95%, gl=2) = 5.9915


### Relaciones estructurales del modelo seleccionado

Una vez elegido el modelo con mejor ajuste, se interpretan sus coeficientes estructurales. En la especificación correcta del modelo parcial se consideran nueve caminos:

- $PA
ightarrow ST$ y $AC
ightarrow ST$: efectos de condiciones laborales sobre satisfacción.
- $PA
ightarrow CO$, $AC
ightarrow CO$ y $ST
ightarrow CO$: efectos sobre compromiso organizacional.
- $PA
ightarrow IP$, $AC
ightarrow IP$, $ST
ightarrow IP$ y $CO
ightarrow IP$: efectos directos sobre intención de permanecer.

Para cada camino se observa el coeficiente, su error estándar, el estadístico $z$ y el valor-p. Si $p<0.05$, se concluye que la relación es estadísticamente significativa al 95% de confianza.

In [7]:
paths=pp[(pp.op=='~')&pp.lval.isin(['ST','CO','IP'])][['lval','rval','Estimate','Est. Std','Std. Err','z-value','p-value']]
print(paths.round(4).to_string(index=False))

lval rval  Estimate  Est. Std  Std. Err   z-value   p-value
  ST   PA    0.1848    0.2373  0.049428  3.739257  0.000185
  ST   AC   -0.0306   -0.0356  0.051809 -0.591183  0.554398
  CO   PA    0.4969    0.4272  0.077823  6.384778       0.0
  CO   AC    0.2505    0.1948  0.069761  3.590861   0.00033
  CO   ST    0.1306    0.0875  0.081605  1.600225  0.109549
  IP   PA    0.1975    0.3540  0.033796  5.843721       0.0
  IP   AC    0.0709    0.1149  0.029954    2.3667  0.017947
  IP   ST    0.0485    0.0678  0.035301  1.374472  0.169295
  IP   CO    0.1576    0.3285   0.02974  5.297898       0.0


## 3.c) Comparación entre hombres y mujeres mediante SEM multigrupo

El enunciado pide evaluar si las relaciones del modelo seleccionado se comportan igual entre hombres y mujeres. Para responderlo de forma más rigurosa, no basta con estimar el modelo por separado y comparar coeficientes uno a uno. Primero debe verificarse si la medición de los constructos es suficientemente comparable entre grupos.

Por eso se sigue una lógica de **invariancia multigrupo**:

1. **Modelo configural:** misma estructura factorial y estructural en hombres y mujeres, pero parámetros libres por grupo.
2. **Invariancia métrica:** se igualan las cargas factoriales entre grupos. Esta etapa es central para comparar relaciones estructurales, porque verifica que los indicadores se relacionen de manera comparable con los constructos.
3. **Invariancia escalar:** se igualan interceptos. Es importante principalmente para comparar medias latentes; no es estrictamente necesaria para comparar caminos estructurales.
4. **Invariancia estructural:** se igualan los caminos estructurales entre hombres y mujeres.
5. **Invariancia estructural parcial:** si la restricción global no es adecuada, se liberan solo los caminos que muestran evidencia clara de diferencia.

La regla principal utilizada es el cambio en CFI: una caída de CFI menor o igual a 0.010 se considera aceptable. También se reporta la diferencia de chi-cuadrado, aunque esta prueba es sensible al tamaño muestral y puede rechazar restricciones pequeñas.


In [8]:
from scipy.optimize import minimize
from scipy.stats import chi2

# Primero se mantienen los modelos separados para obtener los coeficientes descriptivos por género.
grupos = {0: 'Hombres', 1: 'Mujeres'}
est_por_grupo = {}
stats_por_grupo = {}

for g, nombre in grupos.items():
    modelo_g, fit_g, par_g = ajustar(parcial, datos[datos.GENERO == g])
    est_por_grupo[g] = par_g[(par_g.op == '~') & par_g.lval.isin(['ST', 'CO', 'IP'])].copy()
    stats_por_grupo[g] = fit_g
    print(f"{nombre}: n={sum(datos.GENERO == g)}, chi2={fit_g.chi2:.2f}, gl={fit_g.DoF:.0f}, CFI={fit_g.CFI:.3f}, TLI={fit_g.TLI:.3f}, RMSEA={fit_g.RMSEA:.3f}")

filas_z = []
rutas = [('ST','PA'),('ST','AC'),('CO','PA'),('CO','AC'),('CO','ST'),('IP','PA'),('IP','AC'),('IP','ST'),('IP','CO')]
for lhs, rhs in rutas:
    h = est_por_grupo[0][(est_por_grupo[0].lval == lhs) & (est_por_grupo[0].rval == rhs)].iloc[0]
    m = est_por_grupo[1][(est_por_grupo[1].lval == lhs) & (est_por_grupo[1].rval == rhs)].iloc[0]
    z = (h.Estimate - m.Estimate) / np.sqrt(float(h['Std. Err'])**2 + float(m['Std. Err'])**2)
    filas_z.append([f'{rhs}→{lhs}', h.Estimate, m.Estimate, z, 2 * stats.norm.sf(abs(z))])

tabla_z = pd.DataFrame(filas_z, columns=['Relación', 'B hombres', 'B mujeres', 'z descriptivo', 'p descriptivo'])
print('\nComparación descriptiva de coeficientes por grupo')
print(tabla_z.round(4).to_string(index=False))

# -------------------------------------------------------------------------
# Invariancia multigrupo por máxima verosimilitud sobre matrices de covarianza.
# semopy estima grupos separados, pero no impone restricciones multigrupo en esta versión.
# Por eso se implementa directamente la función de ajuste ML para comparar modelos anidados.
# -------------------------------------------------------------------------

latentes = list(constructos)
observadas = obs
p_obs = len(observadas)
q_lat = len(latentes)
G = [0, 1]
Xg = {g: datos[datos.GENERO == g][observadas].values for g in G}
Sg = {g: np.cov(Xg[g], rowvar=False, ddof=0) for g in G}
Mg = {g: Xg[g].mean(axis=0) for g in G}
Ng = {g: len(Xg[g]) for g in G}
logdetS = {g: np.linalg.slogdet(Sg[g])[1] for g in G}

pares_cargas = [(fac, item) for fac, items in constructos.items() for item in items[1:]]
rutas_estructurales = [('ST','PA'),('ST','AC'),('CO','PA'),('CO','AC'),('CO','ST'),('IP','ST'),('IP','CO'),('IP','PA'),('IP','AC')]
ix_lat = {f: i for i, f in enumerate(latentes)}
ix_obs = {v: i for i, v in enumerate(observadas)}

def desempaquetar(v, cargas_iguales, caminos_iguales, escalar=False):
    idx = 0
    cargas_compartidas = {}
    caminos_compartidos = {}
    for par in pares_cargas:
        if par in cargas_iguales:
            cargas_compartidas[par] = v[idx]; idx += 1
    for camino in rutas_estructurales:
        if camino in caminos_iguales:
            caminos_compartidos[camino] = v[idx]; idx += 1

    tau = None
    medias_latentes = {0: np.zeros(q_lat), 1: np.zeros(q_lat)}
    if escalar:
        tau = v[idx:idx+p_obs]; idx += p_obs
        medias_latentes[1] = v[idx:idx+q_lat]; idx += q_lat

    parametros = {}
    for g in G:
        cargas = {}
        for fac, items in constructos.items():
            cargas[(fac, items[0])] = 1.0
        for par in pares_cargas:
            if par in cargas_iguales:
                cargas[par] = cargas_compartidas[par]
            else:
                cargas[par] = v[idx]; idx += 1
        theta_log = v[idx:idx+p_obs]; idx += p_obs
        caminos = {}
        for camino in rutas_estructurales:
            if camino in caminos_iguales:
                caminos[camino] = caminos_compartidos[camino]
            else:
                caminos[camino] = v[idx]; idx += 1
        psi = v[idx:idx+6]; idx += 6
        parametros[g] = (cargas, theta_log, caminos, psi)
    return parametros, tau, medias_latentes

def matrices_grupo(parametros):
    cargas, theta_log, caminos, psi = parametros
    Lambda = np.zeros((p_obs, q_lat))
    for (fac, item), valor in cargas.items():
        Lambda[ix_obs[item], ix_lat[fac]] = valor
    Theta = np.diag(np.exp(theta_log))
    B = np.zeros((q_lat, q_lat))
    for (lhs, rhs), valor in caminos.items():
        B[ix_lat[lhs], ix_lat[rhs]] = valor
    lv_ac, lv_pa, zcov, lr_st, lr_co, lr_ip = psi
    var_ac, var_pa = np.exp(lv_ac), np.exp(lv_pa)
    cov_ac_pa = np.tanh(zcov) * np.sqrt(var_ac * var_pa)
    Psi = np.zeros((q_lat, q_lat))
    Psi[ix_lat['AC'], ix_lat['AC']] = var_ac
    Psi[ix_lat['PA'], ix_lat['PA']] = var_pa
    Psi[ix_lat['AC'], ix_lat['PA']] = Psi[ix_lat['PA'], ix_lat['AC']] = cov_ac_pa
    for fac, lr in zip(['ST','CO','IP'], [lr_st, lr_co, lr_ip]):
        Psi[ix_lat[fac], ix_lat[fac]] = np.exp(lr)
    invB = np.linalg.inv(np.eye(q_lat) - B)
    Sigma = Lambda @ (invB @ Psi @ invB.T) @ Lambda.T + Theta
    return Sigma, Lambda, invB

def objetivo(v, cargas_iguales, caminos_iguales=set(), escalar=False):
    try:
        parametros, tau, medias_latentes = desempaquetar(v, cargas_iguales, caminos_iguales, escalar)
    except Exception:
        return 1e12
    total = 0.0
    for g in G:
        try:
            Sigma, Lambda, invB = matrices_grupo(parametros[g])
            signo, logdet = np.linalg.slogdet(Sigma)
            invSigma = np.linalg.inv(Sigma)
        except Exception:
            return 1e12
        if signo <= 0:
            return 1e12
        F = logdet + np.trace(Sg[g] @ invSigma) - logdetS[g] - p_obs
        if escalar:
            media_predicha = tau + Lambda @ (invB @ medias_latentes[g])
            diferencia = Mg[g] - media_predicha
            F += diferencia @ invSigma @ diferencia
        total += Ng[g] * F
    return total if np.isfinite(total) else 1e12

def valores_iniciales(cargas_iguales, caminos_iguales=set(), escalar=False):
    valores = []
    for par in pares_cargas:
        if par in cargas_iguales:
            valores.append(0.9)
    for camino in rutas_estructurales:
        if camino in caminos_iguales:
            valores.append(0.2)
    if escalar:
        valores += Mg[0].tolist()
        valores += [0.0] * q_lat
    for g in G:
        for par in pares_cargas:
            if par not in cargas_iguales:
                valores.append(0.9)
        valores += [np.log(max(Sg[g][i, i] * 0.4, 0.1)) for i in range(p_obs)]
        for camino in rutas_estructurales:
            if camino not in caminos_iguales:
                valores.append(0.2)
        valores += [0.0, 0.0, np.arctanh(0.3), 0.0, 0.0, 0.0]
    return np.array(valores, dtype=float)

def ajustar_multigrupo(nombre, cargas_iguales, caminos_iguales=set(), escalar=False):
    x0 = valores_iniciales(cargas_iguales, caminos_iguales, escalar)
    mejor = None
    for escala in [1.0, 0.8, 1.2]:
        res = minimize(objetivo, x0 * escala, args=(cargas_iguales, caminos_iguales, escalar), method='L-BFGS-B', options={'maxiter': 5000, 'ftol': 1e-9})
        if mejor is None or res.fun < mejor.fun:
            mejor = res
    momentos = 2 * p_obs * (p_obs + 1) // 2 + (2 * p_obs if escalar else 0)
    gl = momentos - len(mejor.x)
    return {'Modelo': nombre, 'chi2': mejor.fun, 'gl': gl, 'resultado': mejor}

# Estadístico de independencia para CFI/TLI aproximados.
chi_ind, gl_ind = 0.0, 0
for g in G:
    Sigma_ind = np.diag(np.diag(Sg[g]))
    signo, logdet = np.linalg.slogdet(Sigma_ind)
    chi_ind += Ng[g] * (logdet + np.trace(Sg[g] @ np.linalg.inv(Sigma_ind)) - logdetS[g] - p_obs)
    gl_ind += p_obs * (p_obs - 1) // 2

def indices(fila):
    chi_m, gl_m = fila['chi2'], fila['gl']
    cfi = 1 - max(chi_m - gl_m, 0) / max(chi_ind - gl_ind, chi_m - gl_m, 1e-9)
    tli = (chi_ind / gl_ind - chi_m / gl_m) / (chi_ind / gl_ind - 1)
    rmsea = np.sqrt(max((chi_m - gl_m) / (gl_m * (sum(Ng.values()) - 1)), 0))
    fila['CFI'], fila['TLI'], fila['RMSEA'] = cfi, tli, rmsea
    return fila

# Modelos de invariancia.
configural = indices(ajustar_multigrupo('Configural', set()))
metrica_completa = indices(ajustar_multigrupo('Métrica completa', set(pares_cargas)))
# La métrica completa cae levemente por sobre el criterio de ΔCFI; se libera CO3.
metrica_parcial = indices(ajustar_multigrupo('Métrica parcial (libera CO3)', set(pares_cargas) - {('CO','CO3')}))
escalar = indices(ajustar_multigrupo('Escalar completa', set(pares_cargas) - {('CO','CO3')}, escalar=True))
estructural_completa = indices(ajustar_multigrupo('Estructural completa', set(pares_cargas) - {('CO','CO3')}, set(rutas_estructurales)))

filas = []
for modelo, base in [
    (configural, None),
    (metrica_completa, configural),
    (metrica_parcial, configural),
    (escalar, metrica_parcial),
    (estructural_completa, metrica_parcial),
]:
    fila = {k: modelo[k] for k in ['Modelo','chi2','gl','CFI','TLI','RMSEA']}
    if base is not None:
        fila['Δχ2'] = modelo['chi2'] - base['chi2']
        fila['Δgl'] = modelo['gl'] - base['gl']
        fila['p(Δχ2)'] = 1 - chi2.cdf(fila['Δχ2'], fila['Δgl'])
        fila['ΔCFI'] = modelo['CFI'] - base['CFI']
    filas.append(fila)

tabla_invarianza = pd.DataFrame(filas)
print('\nTabla de invariancia multigrupo')
print(tabla_invarianza.round(4).to_string(index=False))


Hombres: n=200, chi2=189.98, gl=160, CFI=0.981, TLI=0.977, RMSEA=0.031
Mujeres: n=200, chi2=213.50, gl=160, CFI=0.972, TLI=0.967, RMSEA=0.041

Comparación descriptiva de coeficientes por grupo
Relación  B hombres  B mujeres  z descriptivo  p descriptivo
   PA→ST     0.2594     0.1443         1.0590         0.2896
   AC→ST    -0.2145     0.3509        -3.3134         0.0009
   PA→CO     0.6348     0.5099         0.7431         0.4574
   AC→CO     0.3620     1.0379        -2.4184         0.0156
   ST→CO     0.0729     0.0830        -0.0587         0.9532
   PA→IP     0.0984     0.2385        -1.9668         0.0492
   AC→IP    -0.0045     0.0038        -0.0724         0.9423
   ST→IP     0.0601     0.0171         0.5756         0.5649
   CO→IP     0.1327     0.2050        -1.1515         0.2495



Tabla de invariancia multigrupo
                      Modelo     chi2  gl    CFI    TLI  RMSEA     Δχ2  Δgl  p(Δχ2)    ΔCFI
                  Configural 403.4729 320 0.9762 0.9717 0.0256     NaN  NaN     NaN     NaN
            Métrica completa 461.4383 335 0.9639 0.9591 0.0308 57.9654 15.0  0.0000 -0.0123
Métrica parcial (libera CO3) 439.3743 334 0.9699 0.9658 0.0281 35.9015 14.0  0.0011 -0.0062
            Escalar completa 492.5408 349 0.9591 0.9554 0.0321 53.1665 15.0  0.0000 -0.0109
        Estructural completa 468.3867 343 0.9642 0.9604 0.0303 29.0124  9.0  0.0006 -0.0057


### Conclusión del análisis multigrupo

El modelo configural presenta buen ajuste, por lo que la estructura general del modelo es plausible tanto en hombres como en mujeres.

La invariancia métrica completa queda levemente por debajo del criterio práctico de cambio en CFI ($\Delta CFI=-0.0123$). Se liberaron sucesivamente restricciones de cargas factoriales y se comparó la mejora de ajuste; la mayor mejora correspondió a permitir que la carga factorial de **CO3** difiriera entre hombres y mujeres. Con esa liberación se obtiene invariancia métrica parcial ($\Delta CFI=-0.0062$), suficiente para comparar los caminos estructurales bajo invariancia métrica parcial.

Desde la métrica parcial se siguen dos ramas analíticas. La primera es la evaluación de invariancia escalar, relevante principalmente para comparar medias latentes. La segunda es la evaluación de invariancia estructural, que compara los caminos entre constructos. La invariancia escalar completa no se sostiene plenamente ($\Delta CFI=-0.0109$). Por tanto, no se realizaron comparaciones de medias latentes entre hombres y mujeres.

La invariancia estructural completa se acepta bajo el criterio principal de $\Delta CFI\leq 0.010$, porque la caída en CFI fue pequeña ($\Delta CFI=-0.0057$). Aunque la diferencia de $\chi^2$ fue significativa, ese estadístico es sensible al tamaño muestral y puede rechazar restricciones pequeñas.

Por lo tanto, las relaciones estructurales pueden considerarse equivalentes entre hombres y mujeres. La única diferencia de medición retenida en el análisis es que se permitió que la carga factorial de CO3 difiriera entre ambos grupos. Esto significa que la relación entre CO3 y compromiso organizacional no tiene exactamente la misma magnitud en hombres y mujeres; sin embargo, no implica que el constructo completo de compromiso organizacional sea diferente entre grupos. Las diferencias numéricas observadas en los coeficientes por grupo se mantienen solo como descripción y no como evidencia de diferencias estructurales significativas.


## 3.d) Explicación y conclusión

El modelo de medición es sólido: el CFA puro de cinco factores correlacionados ajusta muy bien ($\chi^2(160)=220.31$, CFI=0.985, TLI=0.982, RMSEA=0.031) y todas las escalas muestran confiabilidad adecuada (alfa 0.811–0.891; CR 0.811–0.894), AVE superior a 0.50 y discriminación adecuada. La mínima raíz de AVE (0.720) supera la correlación latente máxima (0.562), y el HTMT máximo es 0.569, muy por debajo de 0.85.

El modelo parcialmente mediado representa mejor los datos que el modelo completamente mediado. La comparación arroja $\Delta\chi^2(2)=46.83$, $p<0.001$, y los criterios AIC/BIC calculados sobre $\chi^2$ también favorecen al modelo parcial. Esto indica que fijar en cero los caminos directos $PA→IP$ y $AC→IP$ deteriora significativamente el ajuste.

En el modelo parcial, la percepción del ambiente de trabajo aparece como el antecedente más consistente: se asocia positivamente con satisfacción, compromiso e intención de permanecer. La actitud hacia compañeros se asocia principalmente con compromiso y, en menor medida, con intención de permanecer. El compromiso organizacional también se asocia positivamente con la intención de permanecer. En cambio, satisfacción no presenta asociación adicional significativa con compromiso ni con intención de permanecer una vez incorporadas las demás relaciones.

El análisis multigrupo muestra que la estructura general del modelo es plausible en hombres y mujeres, que se alcanza invariancia métrica parcial y que se acepta invariancia estructural completa bajo el criterio principal de $\Delta CFI\leq 0.010$. Por ello, los caminos estructurales pueden considerarse equivalentes entre hombres y mujeres. Dado que no se alcanzó invariancia escalar completa, no se realizaron comparaciones de medias latentes.

En conjunto, la evidencia respalda el modelo parcialmente mediado como la mejor representación de los datos, con invariancia métrica parcial e invariancia estructural entre hombres y mujeres.
